In [1]:
from functools import partial
from datasets import DropArray
from utils import batch_graph
import torch

/home/mati/anaconda3/envs/drug_prediction_droparray/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print('Loading Dataset and Vectorizing Molecules')
dataset = DropArray("data/droparray_100.pt")

collate_fn = partial(batch_graph, drug_graph_dict=dataset.drug_graph_dict)

train, test = dataset.get_split(how="new_drugs", fold=0)

train_loader = torch.utils.data.DataLoader(
    train, batch_size=128, num_workers=0,
    collate_fn=collate_fn, shuffle=True
)

Loading Dataset and Vectorizing Molecules


  8%|▊         | 2111/27878 [00:05<01:19, 326.16it/s][15:40:48] WARNING: not removing hydrogen atom without neighbors
[15:40:48] WARNING: not removing hydrogen atom without neighbors
[15:40:48] WARNING: not removing hydrogen atom without neighbors
  8%|▊         | 2177/27878 [00:05<01:33, 276.01it/s][15:40:48] WARNING: not removing hydrogen atom without neighbors
[15:40:48] WARNING: not removing hydrogen atom without neighbors
[15:40:49] WARNING: not removing hydrogen atom without neighbors
[15:40:49] WARNING: not removing hydrogen atom without neighbors
[15:40:49] WARNING: not removing hydrogen atom without neighbors
  8%|▊         | 2305/27878 [00:06<01:22, 308.63it/s][15:40:49] WARNING: not removing hydrogen atom without neighbors
[15:40:49] WARNING: not removing hydrogen atom without neighbors
[15:40:49] WARNING: not removing hydrogen atom without neighbors
  9%|▉         | 2627/27878 [00:07<01:14, 338.26it/s][15:40:50] WARNING: not removing hydrogen atom without neighbors
[15:40:5

In [10]:
from model import MoleculeGraphEncoder, DrugCombinationModel

In [7]:
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE   = 128
LR           = 1e-3
EPOCHS       = 50
EMBEDDING_DIM = 128
HIDDEN_DIM   = 256
FOLD         = 0

In [9]:
mol_encoder = MoleculeGraphEncoder(
    node_dim=46,
    hidden_dim=HIDDEN_DIM,
    embedding_dim=EMBEDDING_DIM,
    num_layers=4
)
model = DrugCombinationModel(
    mol_encoder=mol_encoder,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    n_cell_lines=7
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

TypeError: DrugCombinationModel.__init__() got an unexpected keyword argument 'n_cell_lines'

In [8]:
def masked_mse(pred, target, mask):
    loss = ((pred - target) ** 2) * mask
    return loss.sum() / mask.sum()

# ── Train / Eval loop ─────────────────────────────────────────────────────────
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_n = 0.0, 0

    with torch.set_grad_enabled(train):
        for batch in loader:
            batch = batch.to(DEVICE)

            pred, mask = model(batch)           # [B, max_exp], [B, max_exp]
            target = batch.y.view(pred.shape)   # [B, max_exp]

            loss = masked_mse(pred, target, mask)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            n = mask.sum().item()
            total_loss += loss.item() * n
            total_n    += n

    return total_loss / total_n

# ── Main ──────────────────────────────────────────────────────────────────────
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss   = run_epoch(test_loader,  train=False)

    print(f"Epoch {epoch:03d} | train {train_loss:.4f} | val {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        print(f"           ↳ saved (val {val_loss:.4f})")


NameError: name 'model' is not defined